# TII UAV — IMU-Only Characterization + Standalone Mamba

## Goal

This notebook investigates the IMU **on its own**, then asks the important multimodal question:

> **Can IMU provide information that resolves the acoustic ambiguities, especially class 2↔3 and class 3↔4?**

It deliberately does **not** modify or tune the multimodal AeroSync-Mamba architecture.

### We will measure

1. Raw IMU integrity and timing.
2. Accelerometer/gyroscope temporal patterns.
3. Vibration spectra and class-specific frequency structure.
4. Class vs mission vs day effects.
5. Which IMU features distinguish adjacent severities.
6. A standalone real-Mamba IMU classifier.
7. Day-out performance on unseen recording days.
8. Error structure: especially 2↔3 and 3↔4.
9. Audio/IMU complementarity, if the previous audio-only predictions are available.

### Important

The official TII dataset contains accelerometer, gyroscope and audio data for five propeller-damage classes. `SensorCombined` is a synchronized/downsampled IMU stream. The timestamp is used for timing; we do not assume an IMU sample rate from documentation.


## Expected dataset structure

Set `DATA_ROOT` to the directory containing the five class folders:

```text
DATA_ROOT/
  0/
    rosbag2_.../
      ...-SensorCombined.jsonl
  1/
  2/
  3/
  4/
```

The parser is written to handle the common PX4 `SensorCombined` field names:

- `timestamp`
- `accelerometer_m_s2[0:3]`
- `gyro_rad[0:3]`

If your JSONL uses a slightly different nesting/schema, run the schema-inspection cell before changing anything else.


In [ ]:
from pathlib import Path
import json, os, re, math, warnings, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path("/kaggle/input/UAV-Realistic-Fault-Dataset/Dataset")  # CHANGE IF NEEDED
OUT_DIR = Path("/kaggle/working/imu_only")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("exists:", DATA_ROOT.exists())


In [ ]:
# Discover missions and SensorCombined files
mission_rows = []

for class_dir in sorted(DATA_ROOT.iterdir()) if DATA_ROOT.exists() else []:
    if not class_dir.is_dir() or not class_dir.name.isdigit():
        continue
    cls = int(class_dir.name)
    for mission_dir in sorted(class_dir.iterdir()):
        if not mission_dir.is_dir():
            continue
        candidates = list(mission_dir.glob("*-SensorCombined.jsonl"))
        if candidates:
            mission_rows.append({
                "mission": mission_dir.name,
                "class": cls,
                "path": str(candidates[0]),
            })

missions = pd.DataFrame(mission_rows)
print("missions:", len(missions))
print(missions.groupby("class").size())
missions.head()


## 1. Inspect one raw SensorCombined record

Do this before parsing the entire dataset. This prevents silently assuming the wrong JSON schema.


In [ ]:
sample_path = Path(missions.iloc[0]["path"])
with sample_path.open("r") as f:
    first_line = f.readline()

sample_obj = json.loads(first_line)
print(json.dumps(sample_obj, indent=2)[:5000])


In [ ]:
def unwrap_message(obj):
    # Handle common ROS/PX4 JSONL wrappers.
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj

msg = unwrap_message(sample_obj)
print("Top-level keys:", list(msg.keys())[:100])


## 2. Full IMU parser

We retain exactly six physical channels:

`acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`

No audio values are read here.

The timestamp is retained so we can measure the actual IMU sampling interval, gaps and mission duration.


In [ ]:
def get_vec(d, candidates):
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        # Some JSON encodings may flatten [0], [1], [2].
        vals = []
        ok = True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None

def parse_imu_file(path):
    rows = []
    bad = 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, [
                    "accelerometer_m_s2",
                    "accelerometer_m_s2[{i}]",
                    "accelerometer",
                    "acceleration",
                ])
                gyro = get_vec(obj, [
                    "gyro_rad",
                    "gyro_rad[{i}]",
                    "gyro",
                    "angular_velocity",
                ])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1

    arr = np.asarray(rows, dtype=np.float64)
    if len(arr):
        arr = arr[np.argsort(arr[:,0])]
    return arr, bad

arr, bad = parse_imu_file(sample_path)
print("shape:", arr.shape, "bad/unusable lines:", bad)
print("first row:", arr[0] if len(arr) else None)


In [ ]:
# Parse/cache all kept missions
cache_dir = OUT_DIR / "cache"
cache_dir.mkdir(exist_ok=True)

imu_data = {}
integrity = []

for i, r in missions.iterrows():
    p = Path(r["path"])
    cache_path = cache_dir / f"{r['mission']}.npz"
    if cache_path.exists():
        z = np.load(cache_path)
        x = z["x"]
        t = z["t"]
        bad = int(z["bad"])
    else:
        raw, bad = parse_imu_file(p)
        t, x = raw[:,0], raw[:,1:]
        np.savez_compressed(cache_path, t=t, x=x, bad=bad)

    imu_data[r["mission"]] = {"t": t, "x": x, "class": int(r["class"])}

    dt = np.diff(t) / 1e6
    integrity.append({
        "mission": r["mission"],
        "class": int(r["class"]),
        "n": len(t),
        "duration_s": (t[-1]-t[0])/1e6 if len(t)>1 else np.nan,
        "median_dt_ms": np.median(dt)*1e3 if len(dt) else np.nan,
        "p01_dt_ms": np.percentile(dt,1)*1e3 if len(dt) else np.nan,
        "p99_dt_ms": np.percentile(dt,99)*1e3 if len(dt) else np.nan,
        "max_dt_ms": np.max(dt)*1e3 if len(dt) else np.nan,
        "bad_lines": bad,
        "finite": bool(np.isfinite(x).all()),
    })

integrity = pd.DataFrame(integrity)
print(integrity.groupby("class")[["n","duration_s","median_dt_ms","max_dt_ms"]].median())
print("total bad lines:", integrity.bad_lines.sum())
print("all finite:", integrity.finite.all())


## 3. What is the actual IMU timing?

Unlike audio, `SensorCombined` has timestamps. Therefore we can derive the sampling interval directly from the data.

We should **not** carry over the audio uncertainty about 16 kHz here.


In [ ]:
print(integrity[["median_dt_ms","p01_dt_ms","p99_dt_ms","max_dt_ms"]].describe())

plt.figure(figsize=(8,4))
plt.hist(integrity["median_dt_ms"].dropna(), bins=30)
plt.xlabel("mission median IMU interval (ms)")
plt.ylabel("missions")
plt.title("IMU sampling interval across missions")
plt.grid(alpha=.25)
plt.show()


## 4. Representative raw traces

We look at acceleration and angular velocity separately. The key question is whether fault severity changes the vibration pattern, amplitude, or frequency content.


In [ ]:
def plot_trace(mission, start_frac=.40, duration_s=2.0):
    d = imu_data[mission]
    t = (d["t"] - d["t"][0]) / 1e6
    x = d["x"]
    start = int(len(t)*start_frac)
    t0 = t[start]
    keep = (t >= t0) & (t <= t0 + duration_s)

    fig, ax = plt.subplots(2,1,figsize=(12,6),sharex=True)
    for j, lab in enumerate(["acc_x","acc_y","acc_z"]):
        ax[0].plot(t[keep]-t0, x[keep,j], label=lab)
    for j, lab in enumerate(["gyro_x","gyro_y","gyro_z"], start=3):
        ax[1].plot(t[keep]-t0, x[keep,j], label=lab)
    ax[0].set_ylabel("m/s²")
    ax[1].set_ylabel("rad/s")
    ax[1].set_xlabel("time (s)")
    ax[0].legend(ncol=3); ax[1].legend(ncol=3)
    fig.suptitle(f"{mission} | class {d['class']}")
    plt.tight_layout()
    plt.show()

for mission in missions.groupby("class").first()["mission"]:
    plot_trace(mission)


## 5. Steady-flight IMU statistics

Use the same 10–90% mission convention as the audio analysis.

For each mission and channel we calculate:

- RMS / standard deviation
- mean
- peak-to-peak
- high-percentile absolute magnitude

This tells us whether the IMU signal is mainly an amplitude/severity signal.


In [ ]:
def steady_stats(mission):
    d = imu_data[mission]
    x = d["x"]
    n = len(x)
    a,b = int(.10*n), int(.90*n)
    xs = x[a:b]
    out = {"mission":mission, "class":d["class"]}
    for j, name in enumerate(["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]):
        v = xs[:,j]
        out[f"{name}_mean"] = np.mean(v)
        out[f"{name}_std"] = np.std(v)
        out[f"{name}_rms"] = np.sqrt(np.mean(v*v))
        out[f"{name}_p99abs"] = np.percentile(np.abs(v),99)
        out[f"{name}_ptp"] = np.ptp(v)
    return out

imu_stats = pd.DataFrame([steady_stats(m) for m in missions["mission"]])
display(imu_stats.groupby("class").median().round(4))


## 6. IMU vibration spectrum

For each mission we estimate a Welch-like spectrum from the steady-flight signal.

The frequency axis is valid here because the sampling interval comes from the recorded timestamps.

We will inspect both:

- accelerometer vibration
- gyroscope vibration

The important question is whether increasing propeller damage creates stronger/different vibration bands.


In [ ]:
from scipy.signal import welch

def mission_psd(mission, nperseg=2048):
    d = imu_data[mission]
    t = d["t"]
    x = d["x"]
    a,b = int(.10*len(x)), int(.90*len(x))
    xs = x[a:b]

    dt = np.median(np.diff(t[a:b])) / 1e6
    fs = 1.0 / dt
    spectra = []
    freqs = None
    for j in range(6):
        f, p = welch(xs[:,j], fs=fs, nperseg=min(nperseg,len(xs)), detrend="constant")
        freqs = f
        spectra.append(p)
    return freqs, np.asarray(spectra)

# Plot class means
fig, axes = plt.subplots(2,1,figsize=(12,9),sharex=True)
for cls in sorted(missions["class"].unique()):
    arrs = []
    for m in missions.loc[missions["class"]==cls,"mission"]:
        f,p = mission_psd(m)
        arrs.append(p)
    meanp = np.mean(arrs,axis=0)
    for j in range(3):
        axes[0].plot(f, 10*np.log10(meanp[j]+1e-12), alpha=.75, label=f"class {cls}" if j==0 else None)
        axes[1].plot(f, 10*np.log10(meanp[j+3]+1e-12), alpha=.75, label=f"class {cls}" if j==0 else None)
axes[0].set_xlim(0, min(5000, f.max()))
axes[1].set_xlim(0, min(5000, f.max()))
axes[0].set_title("Accelerometer class-mean PSD")
axes[1].set_title("Gyroscope class-mean PSD")
axes[1].set_xlabel("Hz")
axes[0].set_ylabel("log power"); axes[1].set_ylabel("log power")
axes[0].legend()
plt.tight_layout()
plt.show()


## 7. Frequency-band features

To test the physical pattern without committing to a neural architecture, summarize IMU energy into logarithmic frequency bands.

These features are descriptive only.

We will then ask:

- Which channels carry class information?
- Which bands change monotonically with severity?
- Is the signal mainly accelerometer or gyroscope?


In [ ]:
def band_features(mission, bands=None):
    if bands is None:
        bands = [(5,20),(20,50),(50,100),(100,200),(200,500),(500,1000),(1000,2000),(2000,5000)]
    f,p = mission_psd(mission)
    out={"mission":mission,"class":imu_data[mission]["class"]}
    for j,name in enumerate(["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]):
        for lo,hi in bands:
            mask=(f>=lo)&(f<hi)
            out[f"{name}_{lo}_{hi}"] = np.trapz(p[j,mask], f[mask]) if mask.any() else np.nan
    return out

imu_band = pd.DataFrame([band_features(m) for m in missions["mission"]])
print(imu_band.shape)
display(imu_band.groupby("class").median().round(4))


## 8. Class effect vs day effect

Same idea as the audio analysis.

We want to know whether IMU contains:

- a strong fault-class signal,
- a strong recording-day signal,
- or mostly mission-specific variation.

The PCA is descriptive; the variance decomposition is interpreted cautiously because there are only five days.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = [c for c in imu_band.columns if c not in ("mission","class")]
X = imu_band[feature_cols].replace([np.inf,-np.inf],np.nan).fillna(0).values
Xz = StandardScaler().fit_transform(X)

pca = PCA(n_components=2).fit_transform(Xz)
day_map = missions.set_index("mission")["mission"].str.extract(r"rosbag2_(\d{4}_\d{2}_\d{2})")[0]
imu_band["day"] = imu_band["mission"].map(day_map)

fig, ax = plt.subplots(1,2,figsize=(13,5))
for cls in sorted(imu_band["class"].unique()):
    mask=imu_band["class"].values==cls
    ax[0].scatter(pca[mask,0],pca[mask,1],label=f"class {cls}",alpha=.8)
for day in sorted(imu_band["day"].dropna().unique()):
    mask=imu_band["day"].values==day
    ax[1].scatter(pca[mask,0],pca[mask,1],label=day,alpha=.8)
ax[0].set_title("IMU PCA — coloured by class")
ax[1].set_title("IMU PCA — coloured by day")
for a in ax:
    a.set_xlabel("PC1"); a.set_ylabel("PC2"); a.grid(alpha=.2)
ax[0].legend()
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Explained variance:", PCA(n_components=5).fit(Xz).explained_variance_ratio_[:5])


## 9. Adjacent-severity IMU effect sizes

This is one of the most important sections.

We explicitly test:

- 0 vs 1
- 1 vs 2
- 2 vs 3
- 3 vs 4

If IMU separates 2↔3 and 3↔4 using **different features from audio**, it gives us evidence that IMU could rescue acoustic ambiguity.


In [ ]:
from scipy.stats import ttest_ind

def cohens_d(a,b):
    a=np.asarray(a); b=np.asarray(b)
    na,nb=len(a),len(b)
    va,vb=np.var(a,ddof=1),np.var(b,ddof=1)
    sp=np.sqrt(((na-1)*va+(nb-1)*vb)/(na+nb-2))
    return (np.mean(b)-np.mean(a))/sp if sp>0 else np.nan

rows=[]
for c in range(4):
    A=imu_band[imu_band["class"]==c]
    B=imu_band[imu_band["class"]==c+1]
    for feat in feature_cols:
        d=cohens_d(A[feat],B[feat])
        rows.append({"pair":f"{c} vs {c+1}","feature":feat,"d":d,"abs_d":abs(d)})
effects=pd.DataFrame(rows)

for pair in effects["pair"].unique():
    print("\n",pair)
    display(effects[effects["pair"]==pair].sort_values("abs_d",ascending=False).head(15))


## 10. Window-level IMU representation

For the classifier we use timestamp-defined windows rather than assuming an audio sample rate.

Default:

- 2.0 s window
- 1.0 s stride
- 256 uniformly resampled time points
- 6 IMU channels

The model sees temporal IMU structure directly.


In [ ]:
from scipy.interpolate import interp1d

WIN_S = 2.0
STRIDE_S = 1.0
N_RESAMPLE = 256

windows=[]
for _,r in missions.iterrows():
    m=r["mission"]; d=imu_data[m]
    t=(d["t"]-d["t"][0])/1e6
    x=d["x"]
    if len(t)<2 or t[-1]<WIN_S:
        continue

    starts=np.arange(0, t[-1]-WIN_S+1e-9, STRIDE_S)
    for s in starts:
        e=s+WIN_S
        mask=(t>=s)&(t<=e)
        if mask.sum()<10:
            continue
        tt=np.linspace(s,e,N_RESAMPLE)
        xx=np.stack([np.interp(tt,t[mask],x[mask,j]) for j in range(6)],axis=1)
        windows.append({"mission":m,"class":int(r["class"]),"start_s":s,"x":xx})

print("windows:",len(windows))


## 11. Train-only normalization

All normalization is fitted on training missions only within each fold.

This is important because otherwise mission/day information can leak into the test set.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Create day groups
mission_to_day = missions.set_index("mission")["mission"].str.extract(r"rosbag2_(\d{4}_\d{2}_\d{2})")[0].to_dict()
for w in windows:
    w["day"]=mission_to_day[w["mission"]]

W=np.stack([w["x"] for w in windows]).astype(np.float32)
y=np.array([w["class"] for w in windows],dtype=np.int64)
groups=np.array([w["mission"] for w in windows])
days=np.array([w["day"] for w in windows])

print(W.shape, y.shape)


## 12. Real Mamba classifier

This is intentionally simple.

```text
6-channel IMU
    ↓
Linear projection
    ↓
2 real Mamba blocks
    ↓
attention pooling
    ↓
5-class classifier
```

No audio, no LACA, no HTT, no cross-modal fusion.

This gives us a clean answer to:

> **How strong is IMU by itself?**


In [ ]:
try:
    from mamba_ssm import Mamba
    HAVE_MAMBA=True
except Exception as e:
    HAVE_MAMBA=False
    print("mamba_ssm unavailable:",e)

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",DEVICE)

class IMUMamba(nn.Module):
    def __init__(self, in_dim=6, d_model=128, n_layers=2, n_classes=5):
        super().__init__()
        self.proj=nn.Sequential(
            nn.Linear(in_dim,d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )
        self.blocks=nn.ModuleList([
            Mamba(d_model=d_model,d_state=16,d_conv=2,expand=2)
            for _ in range(n_layers)
        ])
        self.norm=nn.LayerNorm(d_model)
        self.pool=nn.Linear(d_model,1)
        self.head=nn.Linear(d_model,n_classes)

    def forward(self,x):
        x=self.proj(x)
        for b in self.blocks:
            x=x+b(x)
        x=self.norm(x)
        a=torch.softmax(self.pool(x).squeeze(-1),dim=1)
        h=(x*a.unsqueeze(-1)).sum(dim=1)
        return self.head(h)

if HAVE_MAMBA:
    model=IMUMamba().to(DEVICE)
    print("parameters:",sum(p.numel() for p in model.parameters()))


## 13. Day-out evaluation

Each recording day is held out completely.

This is the same generalization question used for the audio model:

> Can IMU classify a fault on a day whose missions were never seen during training?


In [ ]:
# Utility functions
def seed_all(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

class WDS(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i],self.y[i]

def train_one(Xtr,ytr,Xv,yv,seed=42,epochs=8):
    seed_all(seed)
    model=IMUMamba().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
    counts=np.bincount(ytr,minlength=5)
    weights=len(ytr)/(5*np.maximum(counts,1))
    lossfn=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
    tr=DataLoader(WDS(Xtr,ytr),batch_size=128,shuffle=True)
    va=DataLoader(WDS(Xv,yv),batch_size=256,shuffle=False)
    best=None; bestf=-1
    for ep in range(epochs):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=lossfn(model(xb),yb)
            loss.backward(); opt.step()
        model.eval()
        ps=[]; ys=[]
        with torch.no_grad():
            for xb,yb in va:
                ps.append(torch.softmax(model(xb.to(DEVICE)),1).cpu().numpy())
                ys.append(yb.numpy())
        p=np.concatenate(ps); yy=np.concatenate(ys)
        f=f1_score(yy,p.argmax(1),average="macro")
        if f>bestf:
            bestf=f
            best=(model.state_dict(),)
    model.load_state_dict(best[0])
    return model

def predict_model(model,X):
    dl=DataLoader(WDS(X,np.zeros(len(X),dtype=np.int64)),batch_size=256,shuffle=False)
    ps=[]
    model.eval()
    with torch.no_grad():
        for xb,_ in dl:
            ps.append(torch.softmax(model(xb.to(DEVICE)),1).cpu().numpy())
    return np.concatenate(ps)


In [ ]:
# Run day-out folds.
# Inner validation = fixed per-class mission holdout, matching the earlier project fallback rule.

unique_days=sorted(pd.unique(days))
seeds=[42,123,2024]
all_pred=[]
fold_summary=[]

for fold_day in unique_days:
    test_missions=set(missions.loc[missions["mission"].map(mission_to_day)==fold_day,"mission"])
    remaining=[m for m in missions["mission"] if m not in test_missions]

    # Fixed inner validation: ~15% per class, seed 2026
    rng=np.random.default_rng(2026)
    val_missions=set()
    for c in range(5):
        cm=[m for m in remaining if int(missions.loc[missions["mission"]==m,"class"].iloc[0])==c]
        k=max(1,int(round(.15*len(cm))))
        val_missions.update(rng.choice(cm,size=k,replace=False).tolist())

    tr_mask=np.array([m not in test_missions and m not in val_missions for m in groups])
    va_mask=np.array([m in val_missions for m in groups])
    te_mask=np.array([m in test_missions for m in groups])

    # Training-only z score
    mu=W[tr_mask].mean(axis=(0,1),keepdims=True)
    sd=W[tr_mask].std(axis=(0,1),keepdims=True)+1e-6
    Xtr=(W[tr_mask]-mu)/sd
    Xv=(W[va_mask]-mu)/sd
    Xte=(W[te_mask]-mu)/sd

    for seed in seeds:
        model=train_one(Xtr,y[tr_mask],Xv,y[va_mask],seed=seed,epochs=8)
        p=predict_model(model,Xte)
        pred=p.argmax(1)
        f=f1_score(y[te_mask],pred,average="macro")
        fold_summary.append({
            "test_day":fold_day,"seed":seed,
            "macro_f1":f,
            "balanced_acc":balanced_accuracy_score(y[te_mask],pred),
            "accuracy":accuracy_score(y[te_mask],pred)
        })
        for idx,prob in zip(np.where(te_mask)[0],p):
            all_pred.append({
                "window_idx":int(idx),
                "mission":groups[idx],
                "class":int(y[idx]),
                "day":days[idx],
                "seed":seed,
                "prob":prob
            })

fold_summary=pd.DataFrame(fold_summary)
display(fold_summary.groupby("test_day")[["macro_f1","balanced_acc","accuracy"]].mean())
display(fold_summary.groupby("seed")["macro_f1"].mean())


## 14. Pooled day-out IMU performance


In [ ]:
# Average probabilities across seeds for each test window
pred_rows=[]
for idx in sorted(set(r["window_idx"] for r in all_pred)):
    rs=[r for r in all_pred if r["window_idx"]==idx]
    prob=np.mean([r["prob"] for r in rs],axis=0)
    pred_rows.append({
        "window_idx":idx,
        "mission":rs[0]["mission"],
        "day":rs[0]["day"],
        "true":rs[0]["class"],
        "pred":int(np.argmax(prob)),
        "prob":prob
    })

pred_df=pd.DataFrame(pred_rows)
print("Window Macro-F1:",f1_score(pred_df.true,pred_df.pred,average="macro"))
print("Balanced accuracy:",balanced_accuracy_score(pred_df.true,pred_df.pred))
print("Accuracy:",accuracy_score(pred_df.true,pred_df.pred))
print(classification_report(pred_df.true,pred_df.pred,digits=3))
print("Confusion matrix:")
print(confusion_matrix(pred_df.true,pred_df.pred))


## 15. Mission-level IMU performance

Multiple windows from the same mission are averaged before classification.
This is directly comparable to the audio mission-level result.


In [ ]:
mission_probs={}
for m,g in pred_df.groupby("mission"):
    probs=np.stack(g["prob"].values)
    mission_probs[m]={
        "true":int(g.true.iloc[0]),
        "pred":int(np.argmax(probs.mean(axis=0))),
        "prob":probs.mean(axis=0)
    }

mp=pd.DataFrame([
    {"mission":m,**{k:v for k,v in d.items() if k!="prob"}}
    for m,d in mission_probs.items()
])
print("Mission Macro-F1:",f1_score(mp.true,mp.pred,average="macro"))
print("Mission balanced accuracy:",balanced_accuracy_score(mp.true,mp.pred))
print("Mission accuracy:",accuracy_score(mp.true,mp.pred))
print(classification_report(mp.true,mp.pred,digits=3))


## 16. Does IMU specifically solve the audio's hard cases?

This is the critical section.

The audio model's main errors were:

- 2 ↔ 3
- 3 ↔ 4

We ask:

1. How often does IMU get an audio mistake correct?
2. How often do both modalities make the same mistake?
3. What is the oracle union accuracy — correct if either modality is correct?

This is **not yet a fusion model**. It is a complementarity diagnostic.


In [ ]:
# Optional: load a previous audio mission-level prediction file if one exists.
AUDIO_CANDIDATES=[
    Path("/kaggle/working/audio_only_mission_predictions.csv"),
    Path("/kaggle/working/audio_only_results.csv"),
]
audio_path=next((p for p in AUDIO_CANDIDATES if p.exists()),None)

if audio_path is None:
    print("No audio prediction table found. Export mission-level audio predictions from the audio notebook,")
    print("then set AUDIO_MISSION_CSV to that path and rerun this section.")
else:
    print("Loading:",audio_path)
    adf=pd.read_csv(audio_path)
    display(adf.head())


In [ ]:
# If your audio file has columns mission,true,pred, rename here if necessary.
if audio_path is not None:
    audio=adf.copy()
    required={"mission","pred"}
    if required.issubset(audio.columns):
        audio["audio_correct"]=audio["pred"]==audio.get("true",audio["pred"])
        merged=mp.merge(audio[["mission","pred","audio_correct"]],on="mission",suffixes=("_imu","_audio"))
        merged["imu_correct"]=merged["pred_imu"]==merged["true"]
        merged["both_correct"]=merged.audio_correct & merged.imu_correct
        merged["audio_only_correct"]=merged.audio_correct & ~merged.imu_correct
        merged["imu_only_correct"]=~merged.audio_correct & merged.imu_correct
        merged["both_wrong"]=~merged.audio_correct & ~merged.imu_correct
        print(merged[["both_correct","audio_only_correct","imu_only_correct","both_wrong"]].mean())
        print("Oracle either correct:", (merged.audio_correct | merged.imu_correct).mean())

        hard=merged[merged["true"].isin([2,3,4])]
        print("\nHard severity subset (true class 2/3/4):")
        print(hard[["both_correct","audio_only_correct","imu_only_correct","both_wrong"]].mean())
    else:
        print("Audio table does not contain the expected mission/pred columns.")


## 17. What to look for

After running the notebook, focus on these outputs:

### A. IMU standalone
Compare its day-out mission Macro-F1 with audio's **0.845**.

### B. IMU class pattern
Look at the IMU confusion matrix.

If IMU also mainly makes:
`2↔3` and `3↔4`,
then it is probably measuring a related severity signal.

If its errors differ from audio's, that is much more interesting.

### C. Complementarity
The most important numbers are:

- `IMU-only correct`
- `Audio-only correct`
- `Both wrong`
- `Oracle either correct`

Especially for true classes 2–4.

### D. Architecture clue

If IMU contains strong vibration-frequency structure, a sensible later fusion design is likely:

```text
Audio → spectral encoder ─┐
                          ├→ fusion → classifier
IMU   → temporal/vibration encoder ─┘
```

rather than forcing both modalities through the same representation.

If IMU's useful information is mostly temporal vibration patterns, then Mamba has a clearer role on the IMU side than it did on the audio side.

### Important stopping rule

Do not change AeroSync-Mamba yet.

First determine **what information IMU actually contains** and whether it complements audio.


## Expected final conclusion format

After execution, summarize the notebook in four lines:

1. **Dominant IMU pattern:** ...
2. **Standalone day-out performance:** ... window / ... mission Macro-F1
3. **Main IMU confusions:** ...
4. **Audio rescue:** IMU correctly resolves ...% of audio's hard errors; oracle either-modality performance = ...

That will tell us whether the multimodal architecture has a real reason to exist before we touch LACA again.
